#  Spark SQL, DataFrames and Datasets Guide

[Official Apache Spark SQL, DataFrames and Datasets Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)

## Learning goals

By the end of this lesson, you should be able to:

- describe how Spark represents tabular data with a schema;
- express the same aggregation with SQL and the DataFrame API;
- explain when Spark executes a transformation and how to inspect its plan;
- choose DataFrames, Datasets, or RDDs for a task.

## Originally

RDD was the primary user-facing API in Spark since its inception.

At the core, an RDD is:

- an immutable distributed collection of elements of your data,
- partitioned across nodes in your cluster that can be operated in parallel
- with a low-level API that offers transformations and actions.

## Is RDD enough ?

:::: {.columns}

::: {.fragment .column width="50%"}
![](https://i.imgflip.com/566dsy.jpg)

[NicsMeme](https://imgflip.com/i/566dsy)
::: 

::: {.fragment .column width="50%"}
![](https://i.imgflip.com/6eibrt.jpg)
:::
::::


## Spark SQL

Spark SQL is Spark's module for structured data processing. It lets us describe work on tabular data using SQL or the DataFrame API.

Because Spark can see the data schema and the operations, it can analyze and optimize the computation. SQL and DataFrame code use the same execution engine.



## Structured data

:::: {.columns}

::: {.fragment .column width="50%"}
**Tidy Dataset**

> “Happy families are all alike; every unhappy family is unhappy in its own way.” –– Leo Tolstoy

> “Tidy datasets are all alike, but every messy dataset is messy in its own way.” –– Hadley Wickham

<https://uta.pressbooks.pub/datanotebook/chapter/1-3-structured-data/>
::: 

::: {.fragment .column width="50%"}
![](https://uta.pressbooks.pub/app/uploads/sites/85/2021/05/Tidy-Data.png)
:::
::::


## Structured data: a concrete example

| order_id | category | amount |
|---:|---|---:|
| 1 | books | 20.0 |
| 2 | stationery | 15.0 |

Each row is a record; each named column has a defined type (for example, `order_id: integer`, `category: string`, `amount: double`). Structured data may contain text or nested fields too—not only numbers.

This connects to the [Schema Registry lesson](Schema%20Registry.ipynb): a registered schema defines the fields and types that producers and consumers agree on. Spark's DataFrame schema describes the columns Spark sees when it processes the data. They are related descriptions of structure, but not the same thing: the registry versions and checks a shared data contract; Spark uses the DataFrame schema to interpret and query the data.

## Dataset and DataFrame

A **Dataset** is Spark's distributed collection abstraction. In Scala and Java, `Dataset[T]` can hold objects of a specific type `T`, checked at compile time.

| Typed Dataset | DataFrame |
|---|---|
| `Dataset[Person]` | `Dataset[Row]` in Scala/Java |
| Records use a JVM class, such as `Person` | Rows share named columns and a schema |
| Typed Dataset API: Scala and Java | DataFrame API: Scala, Java, Python, and R |

So, a DataFrame is a **special case of Dataset** in Scala and Java: it is `Dataset[Row]`, where each row follows the DataFrame's schema. Python and R provide the DataFrame API, but not the typed `Dataset[T]` API.

## DataFrame: the tabular Dataset

A DataFrame is a distributed table with named columns and a schema (column names and types). In Scala and Java, it is `Dataset[Row]`; each row's fields are described by that shared schema.

- It can be created from files, tables, databases, or existing collections.

- The DataFrame API is available in Scala, Java, Python, and R.

**Next:** compare SQL and DataFrame code for the same task.

## A DataFrame has a schema

The schema names each column and describes its type and whether it can be null. For example:

```text
root
 |-- order_id: integer (nullable = false)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
```

Null means a value is missing or unknown; it is not the same as zero or an empty string. Use `printSchema()` to inspect a DataFrame's schema.

### Like a spreadsheet - but not the same

:::: {.columns}

::: {.fragment .column width="50%"}
- A simple analogy is that a DataFrame is like a spreadsheet with named columns. 
- A Spark DataFrame is distributed as partitions and can be processed across a cluster; it can also run locally.
- This makes the same table-style operations usable on data larger than one machine can efficiently handle.
::: 

::: {.fragment .column width="50%"}
![](https://upload.wikimedia.org/wikipedia/commons/7/7a/Visicalc.png)

:::
::::


### But distributed 

:::: {.columns}

::: {.fragment .column width="50%"}
Spark splits data into partitions. Tasks process partitions in parallel, and operations such as joins or aggregations may exchange data between machines (a shuffle). A cluster is useful for data or computations that benefit from this parallelism—but it adds coordination overhead.
::: 

::: {.fragment .column width="50%"}
![](https://www.databricks.com/wp-content/uploads/2018/05/DataFrames.png)
:::
::::





## Evolution

::: {.r-stack}
![](images/human-evolution-monkey-modern-man-programmer-computer-user-isolated-white_33099-1593.jpg){.fragment}

![](https://databricks.com/wp-content/uploads/2016/06/Unified-Apache-Spark-2.0-API-1.png){.fragment}
:::

## A nice comparison

![](images/Apache-Spark-RDD-vs-DataFrame-vs-DataSet-1.jpg)

<https://data-flair.training/blogs/apache-spark-rdd-vs-dataframe-vs-dataset/>

## Which API should I use?

- **DataFrame / SQL:** the default choice for structured or semi-structured data, joins, filters, and aggregations. Spark can optimize these operations.
- **Dataset:** use Scala or Java when compile-time typing of JVM objects is useful; in those languages, a DataFrame is `Dataset[Row]`.
- **RDD:** use when you specifically need lower-level control or transformations that do not fit the structured APIs. Unstructured input alone does not require an RDD—you can often parse it into a DataFrame.

Prefer the highest-level API that expresses the task clearly.

## Choosing an API: quick recap

- In Python and R, use DataFrames for most Spark SQL work.
- In Scala and Java, use DataFrames for relational operations; choose typed Datasets when their compile-time type safety helps.
- Use RDDs only when lower-level control is an actual requirement—not just because the input started as text or another unstructured format.

## Spark SQL: one engine, multiple interfaces

- **Spark SQL** is Spark's module for structured data processing.
- Use SQL or the DataFrame API to describe a computation; Spark builds an optimized execution plan.
- Both interfaces use the same underlying engine, so choose the expression style that best fits the task.

# The SQL interface{background-color="white" background-image="https://github.com/nicshub/sdsdbms/blob/main/images/sqladam.jpg?raw=true" background-size="50%" background-opacity="0.8"}

## Writing SQL queries in Spark

Spark SQL is the structured-data module; this section focuses on its **SQL language interface**. It is one way to express queries, alongside the DataFrame API.

- Run SQL against tables or temporary views registered in Spark.
- From Python, Scala, or Java, `spark.sql(...)` returns a DataFrame (a `Dataset[Row]` in Scala/Java).
- Spark can also integrate with a Hive metastore to access its tables.
- SQL clients can connect through JDBC/ODBC; Spark also provides command-line tools.

## First query: SQL

Create a small DataFrame and register it as a temporary view so Spark SQL can query it:

```python
sales = spark.createDataFrame(
    [("books", 20.0), ("stationery", 15.0), ("books", 5.0)],
    ["category", "amount"]
)
sales.createOrReplaceTempView("sales")
```

```sql
SELECT category, SUM(amount) AS total
FROM sales
GROUP BY category
ORDER BY category
```

Run the query from PySpark and display the result:
```python
spark.sql("""
SELECT category, SUM(amount) AS total
FROM sales
GROUP BY category
ORDER BY category
""").show()
```

Expected result:
```text
+----------+-----+
|  category|total|
+----------+-----+
|     books| 25.0|
|stationery| 15.0|
+----------+-----+
```

## Same computation: DataFrame API

The DataFrame API can express the same aggregation. It returns the same rows as the SQL query:

```python
from pyspark.sql import functions as F

result = sales.groupBy("category").agg(
    F.sum("amount").alias("total")
).orderBy("category")

result.show()
```

Choose SQL or the DataFrame API based on which makes the transformation clearer.

## Transformations are lazy

```python
large_sales = sales.filter(sales.amount > 10).select("category", "amount")
large_sales.explain()  # inspect the plan
large_sales.show()     # action: compute and display rows
```

`filter` and `select` describe transformations; Spark can optimize them together. An action such as `show`, `count`, or `write` triggers computation. `explain()` displays the plan without collecting the result rows.

## Why can Spark optimize DataFrames?

- SQL and DataFrame operations form a logical plan that Spark can analyze and optimize before execution.
- For example, Spark can push filters closer to the data source and avoid reading unused columns when the source supports it.
- Optimization can reduce work, but it does not guarantee every DataFrame job is faster; data layout, shuffles, and the workload still matter.

Use `explain()` to inspect the plan for a query.

# Learn by Examples

## Demos with Spark Docker
<https://spark.apache.org/docs/latest/sql-getting-started.html>

::: {..fragment}
[Code](https://github.com/apache/spark/blob/master/examples/src/main/python/sql/basic.py)
```bash 
docker run --hostname spark -p 4040:4040 -it --rm --workdir /opt/spark  apache/spark /opt/spark/bin/spark-submit examples/src/main/python/sql/basic.py
```
:::

::: {.fragment}
[Code](https://github.com/apache/spark/blob/master/examples/src/main/java/org/apache/spark/examples/sql/JavaSparkSQLExample.java)
```bash
docker run --hostname spark -p 4040:4040 -it --rm --workdir /opt/spark  apache/spark /opt/spark/bin/spark-submit --class org.apache.spark.examples.sql.SparkSQLExample --master local examples/jars/spark-examples.jar
```
:::




## Demos with notebooks

Databricks

- [Olympics Databricks](https://dbc-47bba2d3-5b65.cloud.databricks.com/editor/notebooks/1359480913523914?o=896987012068068)
- [Flight Databricks](https://databricks-prod-cloudfront.cloud.databricks.com/public/4027ec902e239c93eaaa8714f173bcfc/1408031979081866/4241690966276695/2956912205716139/latest.html)



## Official Apache Spark references

- [Spark SQL, DataFrames and Datasets Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html) — concepts, SQL, DataFrames, Datasets, and Hive integration.
- [Spark SQL Getting Started](https://spark.apache.org/docs/latest/sql-getting-started.html) — SparkSession, creating DataFrames, and running SQL.
- [PySpark DataFrame API reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.html) — methods and behavior of DataFrames.
- [PySpark SparkSession API reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.html) — creating sessions and running SQL programmatically.
- [RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html) — RDD concepts, transformations, and actions.

These links point to the latest Spark documentation; select the documentation version matching the Spark release used in class.